In [1]:
import os
import pathlib
import sys
import time
import warnings

import pandas as pd
import psutil
import tomli
from image_analysis_3D.file_utils.arg_parsing_utils import (
    check_for_missing_args,
    parse_args,
)
from image_analysis_3D.file_utils.notebook_init_utils import (
    bandicoot_check,
    init_notebook,
)

root_dir, in_notebook = init_notebook()

from image_analysis_3D.featurization_utils.feature_writing_utils import (
    format_morphology_feature_name,
    save_features_as_parquet,
)
from image_analysis_3D.featurization_utils.loading_classes import (
    ImageSetLoader,
    ObjectLoader,
)
from image_analysis_3D.featurization_utils.resource_profiling_util import (
    start_profiling,
    stop_profiling,
)
from image_analysis_3D.featurization_utils.texture_utils import measure_3D_texture

profile_base_dir = bandicoot_check(
    pathlib.Path(os.path.expanduser("~/mnt/bandicoot/NF1_organoid_data")).resolve(),
    root_dir,
)

warnings.filterwarnings(
    "ignore",
    message="invalid escape sequence",
    category=SyntaxWarning,
    module="mahotas",
)

In [2]:
if not in_notebook:
    arguments_dict = parse_args()
    patient = arguments_dict["patient"]
    well_fov = arguments_dict["well_fov"]
    channel = arguments_dict["channel"]
    compartment = arguments_dict["compartment"]
    processor_type = arguments_dict["processor_type"]
    input_subparent_name = arguments_dict["input_subparent_name"]
    mask_subparent_name = arguments_dict["mask_subparent_name"]
    output_features_subparent_name = arguments_dict["output_features_subparent_name"]

else:
    # well_fov = "C9-7"
    # patient = "NF0035_T1"
    # channel = "Mito"
    # compartment = "Cell"
    well_fov = "C4-2"
    patient = "NF0014_T1"
    channel = "DNA"
    compartment = "Cell"
    processor_type = "CPU"
    input_subparent_name = "zstack_images"
    mask_subparent_name = "segmentation_masks"
    output_features_subparent_name = "extracted_features"

image_set_path = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{input_subparent_name}/{well_fov}/"
)
mask_set_path = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{mask_subparent_name}/{well_fov}/"
)
output_parent_path = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{output_features_subparent_name}/{well_fov}/"
)
output_parent_path.mkdir(parents=True, exist_ok=True)
channel_mapping_file_path = pathlib.Path(
    f"{root_dir}/config/channel_mapping.toml"
).resolve(strict=True)

In [3]:
# read in channel mapping
with open(channel_mapping_file_path, "rb") as f:
    channel_mapping_dict = tomli.load(f)
channel_n_compartment_mapping = channel_mapping_dict["channel_mapping"]

In [4]:
start_time, start_mem = start_profiling()

In [5]:
image_set_loader = ImageSetLoader(
    image_set_path=image_set_path,
    mask_set_path=mask_set_path,
    anisotropy_spacing=(1, 0.1, 0.1),
    channel_mapping=channel_n_compartment_mapping,
    image_set_name=well_fov,
    mask_key_name=[channel_n_compartment_mapping[compartment]],
    raw_image_key_name=[channel_n_compartment_mapping[channel]],
)

In [6]:
object_loader = ObjectLoader(
    image_set_loader.image_set_dict[channel],
    image_set_loader.image_set_dict[compartment],
    channel,
    compartment,
)
output_texture_dict = measure_3D_texture(
    object_loader=object_loader,
    distance=3,  # distance in pixels 3 is what CP uses
)
final_df = pd.DataFrame(output_texture_dict)

final_df = final_df.pivot(
    index="object_id",
    columns="texture_name",
    values="texture_value",
)
final_df.reset_index(inplace=True)
final_df.rename(
    columns={
        col: format_morphology_feature_name(
            compartment=compartment,
            channel=channel,
            feature_type="Texture",
            measurement=col,
        )
        if col != "object_id"
        else col
        for col in final_df.columns
    },
    inplace=True,
)
final_df.insert(0, "image_set", image_set_loader.image_set_name)
final_df.columns.name = None

save_path = save_features_as_parquet(
    parent_path=output_parent_path,
    df=final_df,
    feature_type="Texture",
    channel=channel,
    compartment=compartment,
    cpu_or_gpu=processor_type,
)
final_df.head()

,image_set,object_id,Cell_DNA_Texture_AngularSecondMoment-3-00-256,Cell_DNA_Texture_AngularSecondMoment-3-01-256,Cell_DNA_Texture_AngularSecondMoment-3-02-256,Cell_DNA_Texture_AngularSecondMoment-3-03-256,Cell_DNA_Texture_AngularSecondMoment-3-04-256,Cell_DNA_Texture_AngularSecondMoment-3-05-256,Cell_DNA_Texture_AngularSecondMoment-3-06-256,Cell_DNA_Texture_AngularSecondMoment-3-07-256,...,Cell_DNA_Texture_Variance-3-03-256,Cell_DNA_Texture_Variance-3-04-256,Cell_DNA_Texture_Variance-3-05-256,Cell_DNA_Texture_Variance-3-06-256,Cell_DNA_Texture_Variance-3-07-256,Cell_DNA_Texture_Variance-3-08-256,Cell_DNA_Texture_Variance-3-09-256,Cell_DNA_Texture_Variance-3-10-256,Cell_DNA_Texture_Variance-3-11-256,Cell_DNA_Texture_Variance-3-12-256
0,C4-2,257,2.339816e-315,0.0,1.0,0.0,111.741573,146.123596,72.313433,57.089552,...,77.359551,0.0,0.0,0.0,45.671642,5.288015e-91,1.307846e-163,4.471727e-91,0.0,0.0
1,C4-2,514,6.936646e-310,0.0,1.0,0.0,111.741573,163.314607,76.119403,49.477612,...,74.494382,0.0,0.0,0.0,72.313433,2.420922e-322,2.843303e-173,9.698584e-101,0.0,0.0
2,C4-2,771,2.339816e-315,0.0,1.0,0.0,111.741573,154.719101,60.895522,57.089552,...,74.494382,0.0,0.0,0.0,57.089552,0.000000e+00,8.850032e-159,1.424530e-105,0.0,0.0
3,C4-2,1028,2.339816e-315,0.0,1.0,0.0,100.280899,143.258427,72.313433,53.283582,...,74.494382,0.0,0.0,0.0,68.507463,0.000000e+00,1.929630e-168,3.065436e-115,0.0,0.0
4,C4-2,1799,0.000000e+00,0.0,1.0,0.0,123.202247,148.988764,87.537313,57.089552,...,71.629213,0.0,0.0,0.0,53.283582,1.118952e+60,8.848916e-159,6.545204e-125,0.0,0.0


In [7]:
stop_profiling(
    start_time=start_time,
    start_mem=start_mem,
    feature_type="Texture",
    well_fov=well_fov,
    patient_id=patient,
    channel=channel,
    compartment=compartment,
    CPU_GPU="CPU",
    output_file_dir=pathlib.Path(
        f"{root_dir}/data/{patient}/extracted_features/run_stats/{well_fov}_{channel}_{compartment}_Texture_CPU.parquet"
    ),
)


        Memory and time profiling for the run:
        Patient ID: NF0014_T1
        Well and FOV: C4-2
        Feature type: Texture
        CPU/GPU: CPU
        Peak memory (tracemalloc): 447.06 MB
        Current memory (tracemalloc): 306.81 MB
        RSS at end: 499.88 MB
        Time elapsed:
        --- 13.64 seconds ---
        --- 0.23 minutes ---
        --- 0.00 hours ---
    


True